In [ ]:
import sys
import os


In [ ]:


import cv2
from ultralytics import YOLO
import mediapipe as mp
import csv
# Now you can import your custom functions
from src.pose_detector import draw_annotations, extract_and_save_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# --- SETUP ---
model = YOLO('runs/pose/train2/weights/best.pt')
video_path = "data/raw/sample10.mp4" # Use your structured path
cap = cv2.VideoCapture(video_path)

# Video Writer Setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('data/processed/output_analysis.mp4', fourcc, fps, (width, height))

# CSV Setup
csv_file = open('data/processed/swing_data.csv', 'w', newline='')
writer = csv.writer(csv_file)

# Create CSV Header (Frame + 33 MediaPipe Landmarks * 4 attributes)
header = ['frame']
for i in range(33):
    header.extend([f'mp_{i}_x', f'mp_{i}_y', f'mp_{i}_z', f'mp_{i}_vis'])
header.append('yolo_keypoints_raw') # Add more specific headers for your YOLO model later
writer.writerow(header)

# --- MAIN LOOP ---
mp_pose = mp.solutions.pose
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    frame_count = 0
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
            
        # 1. Inference
        yolo_results = model(frame, verbose=False) # verbose=False keeps terminal clean
        
        # MediaPipe needs RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_rgb.flags.writeable = False
        mp_results = pose.process(image_rgb)
        
        # 2. Draw Visuals (Using your new function)
        final_frame = draw_annotations(frame, yolo_results, mp_results)
        
        # 3. Save Data (Using your new function)
        extract_and_save_data(writer, frame_count, mp_results, yolo_results)
        
        # 4. Display/Write
        out.write(final_frame)
        cv2.imshow('Golf Swing AI', final_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
        frame_count += 1

# Cleanup
cap.release()
out.release()
csv_file.close() # Important: Close the CSV file!
cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/swing_data.csv'